# Step 5 : backward passによる入力修正則の導出

## 概要

Step4で導出したQ関数の二次近似から、入力修正則を導出し、それを用いた価値関数の更新から、backward passの計算を導出する。

## 1. 入力修正則の導出

Step4ではQ関数の近傍を次の二次近似式として$Q_n$のように求まった。

$$
\begin{aligned}
Q_n(\bar{X}_n + \delta X_n, \bar{u}_n+ \delta u_n) =& \ell_n(\bar{X}_n + \delta X_n, \bar{u}_n+ \delta u_n) + V_{n+1}(\bar{X}_{n+1} + \delta X_{n+1}) \\
\approx & \bar{Q}_n + Q_{X,n}^T \delta X_n + Q_{u,n}^T \delta u_n \\
& + \frac{1}{2} \delta X_n^T Q_{XX,n} \delta X_n + \delta u_n^T Q_{uX,n} \delta X_n + \frac{1}{2} \delta u_n^T Q_{uu,n} \delta u_n
\end{aligned}
$$

$$
\begin{aligned}
\bar{Q}_n =& \ell_n(\bar{X}_n, \bar{u}_n) + \bar{V}_{n+1} \\
Q_{X,n} =& \ell_{X,n} + A_n^T V_{X,n+1} \\
Q_{u,n} =& \ell_{u,n} + B_n^T V_{X,n+1} \\
Q_{XX,n} =& \ell_{XX,n} + A_n^T V_{XX, n+1} A_n \\
Q_{uX,n} =& \ell_{uX,n} + B_n^T V_{XX, n+1} A_n \\
Q_{uu,n} =& \ell_{uu,n} + B_n^T V_{XX, n+1} B_n
\end{aligned}
$$


この$Q_n$を$\delta u_n$で偏微分してゼロと置き、$\delta u_n$を求めることで、それが時刻$n$における入力修正則になることを示す。

まず、$\delta u_n$で偏微分を行うため、$\delta u_n$に関する項でまとめると次のようになる。$C$は$\delta u_n$に依存しない項である。

$$
Q_n(\delta u_n) = (Q_{u,n} + Q_{uX,n} \ \delta X_n)^T \delta u_n + \frac{1}{2}\delta u_n^T \ Q_{uu,n} \ \delta u_n + C 
$$

この$Q_n(\delta u_n)$を$\delta u_n$で偏微分すると、以下となる。

$$
\frac{\partial Q_n}{\partial \delta u_n} = Q_{u,n} + Q_{uX,n} \ \delta X_n + Q_{uu,n} \ \delta u_n
$$

最小点の候補は以下のように、一階偏微分の傾きがゼロになる停留条件を満たす$\delta u_n$である。

$$
\begin{aligned}
\frac{\partial Q_n}{\partial \delta u_n} &= Q_{u,n} + Q_{uX,n} \ \delta X_n + Q_{uu,n} \ \delta u_n = 0 \\
& \downarrow \\
\delta u_n &= -Q_{uu,n}^{-1} \ Q_{u,n} - Q_{uu,n}^{-1} \ Q_{uX,n} \ \delta X_n 
\end{aligned}
$$

そのような$\delta u_n$を入力修正則 $\delta u_n^\star$とする。

$$
\boxed{
\delta u_n^\star =  k_n + K_n\ \delta X_n 
}
$$

$$
\boxed{
k_n = -Q_{uu,n}^{-1} \ Q_{u,n}, \quad K_n = - Q_{uu,n}^{-1} \ Q_{uX,n}
}
$$

この入力修正則は、状態摂動$\delta X_{n}$に対して、時刻$n$のステージコストと、時刻$n+1$から終端までの将来コストを合わせた局所二次Q関数が最小になるように、時刻$n$の入力をどれだけ修正するかを計算する。

入力修正則により、以下の２項目の和である局所二次Q関数を最小化する。

$$
Q_n = \underbrace{\ell_n}_{\text{時刻}n\text{のコスト}} + \underbrace{V_{n+1}}_{\text{時刻}n+1\text{以降の最小コスト}}
$$

backward passでは、$\delta X_n$に具体的な値を与えるのではなく、$\delta X_n$を変数として残したまま、任意の状態摂動に対する入力修正則を構成する$k_n, K_n$を計算する。


この入力修正則が、時刻$n$のステージコストと時刻$n+1$以降の価値関数から構成された局所二次Q関数を最小化することを確認する。

それには、$Q_n$の$\delta u_n$によるHessianが正定値であることを確認すればよい。

つまり、次のように$Q_{uu,n}$が正定値であることが、一意な最小点となるための十分条件である。この場合、$Q_n$は$\delta u_n$に関して狭義凸関数となり、一意な最小点が求まる。

$$
\frac{\partial^2 Q_n}{\partial \delta u_n^2} = Q_{uu,n} \succ 0
$$

$Q_{uu,n}$は以下で構成される。

$$
Q_{uu,n} = \ell_{uu,n} + B_n^T V_{XX, n+1} B_n
$$

よって、

$$
\ell_{uu,n} \succ 0 , \quad V_{XX, n+1} \succeq 0
$$

なら半正定値行列の性質より、以下となる。

$$
B_n^T V_{XX, n+1} B_n \succeq 0
$$

よって、$Q_{uu,n}$が正定値行列であることが保証される。

$$
Q_{uu,n} \succ 0
$$

しかし、非線形で非凸な問題では、局所二次近似によって得られる$V_{XX,n+1}$やコストのHessianが不定になることがあり、$Q_{uu,n}$が正定値であることは必ずしも保証されない。この問題は Step8 の数値安定化で扱う。このNoteでは$Q_{uu,n}$が正定値であると仮定して議論を進める。


## 2. 価値関数の更新式

上記の入力修正則の係数($k_n, K_n$)は($Q_{uu,n},Q_{u,n},Q_{uX,n}$)で構成されており、($Q_{uu,n},Q_{u,n},Q_{uX,n}$)は価値関数($V_{XX,n+1}, V_{X, n+1}$)から構成されている。よって以下の流れがある。

$$
(V_{XX,n+1}, V_{X, n+1}) \rightarrow (Q_{uu,n},Q_{u,n},Q_{uX,n}) \rightarrow (k_n, K_n)
$$

次は($k_n,K_n$)により、($V_{XX,n}, V_{X, n}$)が求まることを示す。

入力修正則 $\delta u_n^\star = k_n + K_n \delta X_n$ を局所二次Q関数 $\tilde{Q}_n$に代入する。$\tilde{Q}_n$は$\delta u_n^\star$により最小化されているため、次のように価値関数$V_n$の二次近似$\tilde{V}_n$と等式として構成することが出来る。

$$
\begin{aligned}
\tilde{V}_n(\bar{X}_n + \delta X_n) &= \underset{\delta u_n}{\min} \tilde{Q}_n(\bar{X} + \delta X_n , \bar{u}_n + \delta u_n) \\
\tilde{V}_n(\bar{X}_n + \delta X_n) &= \tilde{Q}_n(\bar{X} + \delta X_n , \bar{u}_n + \delta u_n^\star)
\end{aligned}
$$

左辺の価値関数の二次近似式$\tilde{V}_n$は以下である。

$$
\tilde{V}_n(\bar{X}_n + \delta X_n) = \bar{V}_{n} + V_{X, n}^T \delta X_{n} + \frac{1}{2} \delta X_{n}^T \ V_{XX, n} \ \delta X_{n}
$$

右辺のQ関数の二次近似式$\tilde{Q}_n$は以下である。

$$
\begin{aligned}
\tilde{Q}_n(\bar{X}_n + \delta X_n, \bar{u}_n+ \delta u_n^\star) 
= & \bar{Q}_n  + \left( Q_{u,n} + Q_{uX,n} \delta X_n \right)^T \delta u_n^\star + \frac{1}{2} (\delta {u_n^{\star}})^T Q_{uu,n} \delta u_n^\star \\
& + Q_{X,n}^T \delta X_n + \frac{1}{2} \delta X_n^T Q_{XX,n} \delta X_n 
\end{aligned}
$$

$\delta u_n^\star$に関する項目だけを展開すると、以下のようにまとまる。

$$
\begin{aligned}
 & \left( Q_{u,n} + Q_{uX,n} \delta X_n \right)^T (k_n + K_n \delta X_n) + \frac{1}{2} (k_n + K_n \delta X_n)^T Q_{uu,n} (k_n + K_n \delta X_n) \\
 &= Q_{u,n}^T \ k_n + Q_{u,n}^T \ K_n \delta X_n + k_n^T Q_{uX,n} \delta X_n + \delta X_n ^T Q_{uX,n}^T K_n \delta X_n \\
 & \space \space +  \frac{1}{2} k_n^T Q_{uu,n} k_n + \frac{1}{2} k_n^T Q_{uu,n}  K_n \delta X_n + \frac{1}{2} \delta X_n^T K_n^T  Q_{uu,n} k_n + \frac{1}{2} \delta X_n^T K_n^T  Q_{uu,n} K_n \delta X_n \\
 &= Q_{u,n}^T \ k_n + \frac{1}{2} k_n^T Q_{uu,n} k_n \\
 & \space \space + \left( K_n^T Q_{u,n} \   + Q_{uX,n}^T k_n + K_n^T  Q_{uu,n} k_n\right)^T \delta X_n \\
 & \space \space + \delta X_n^T Q_{uX,n}^T K_n \delta X_n  + \frac{1}{2} \delta X_n^T K_n^T  Q_{uu,n} K_n \delta X_n
\end{aligned}
$$

２行目の式から３行目の式では、以下の２項はともにスカラーであり、$Q_{uu,n}$は対称なので、一つにまとめられる関係を用いている。

$$
\frac{1}{2} k_n^T Q_{uu,n}  K_n \delta X_n + \frac{1}{2} \delta X_n^T K_n^T  Q_{uu,n} k_n
$$


二次近似式 $\tilde{Q}_n$の項で、定数項、$\delta X_n$の一次項、二次項でまとめると次のようになる。

- 定数項

$$
\bar{Q}_n + Q_{u,n}^T \ k_n + \frac{1}{2} k_n^T Q_{uu,n} k_n
$$

- 一次項

$$

Q_{X,n}^T \delta X_n + \left( K_n^T Q_{u,n} \   + Q_{uX,n}^T k_n + K_n^T  Q_{uu,n} k_n  \right)^T \delta X_n \\
= \left(Q_{X,n} + K_n^T Q_{u,n} \   + Q_{uX,n}^T k_n + K_n^T  Q_{uu,n} k_n \right)^T \delta X_n
$$

- 二次項

$$
\begin{aligned}
& \frac{1}{2} \delta X_n^T Q_{XX,n} \delta X_n  + \delta X_n^T Q_{uX,n}^T K_n \delta X_n  + \frac{1}{2} \delta X_n^T K_n^T  Q_{uu,n} K_n \delta X_n \\
& = \frac{1}{2} \delta X_n^T Q_{XX,n} \delta X_n  + \frac{1}{2} \delta X_n^T (Q_{uX,n}^T K_n + K_n^T Q_{uX,n} )\delta X_n  + \frac{1}{2} \delta X_n^T K_n^T  Q_{uu,n} K_n \delta X_n \\
& = \frac{1}{2} \delta X_n^T \left(Q_{XX,n} + Q_{uX,n}^T K_n + K_n^T Q_{uX,n} + K_n^T  Q_{uu,n} K_n \right) \delta X_n 
\end{aligned}
$$

$k_n, K_n$の定義から得られる次の関係を用いて、各項を整理する。

$$
Q_{uu,n} k_n = - Q_{u,n}, \quad Q_{uu,n} K_n = - Q_{uX,n}
$$

- 定数項

$$
\bar{Q}_n + Q_{u,n}^T \ k_n - \frac{1}{2} k_n^T Q_{u,n} = \bar{Q}_n + \frac{1}{2} Q_{u,n}^T \ k_n
$$

- 一次項

$$
\begin{aligned}
&\left(Q_{X,n} + K_n^T Q_{u,n}  + Q_{uX,n}^T k_n + K_n^T  Q_{uu,n} k_n \right)^T \delta X_n \\
&= \left(Q_{X,n} + \cancel{K_n^T Q_{u,n}}  + Q_{uX,n}^T k_n - \cancel{K_n^T Q_{u,n} }\right)^T \delta X_n \\
&= \left(Q_{X,n} + Q_{uX,n}^T k_n \right)^T \delta X_n
\end{aligned}
$$

- 二次項

$$
\begin{aligned}
& \frac{1}{2} \delta X_n^T \left(Q_{XX,n} + Q_{uX,n}^T K_n + K_n^T Q_{uX,n} + K_n^T  Q_{uu,n} K_n \right) \delta X_n \\
& = \frac{1}{2} \delta X_n^T \left(Q_{XX,n} + Q_{uX,n}^T K_n + \cancel{K_n^T Q_{uX,n}} - \cancel{K_n^T Q_{uX,n}} \right) \delta X_n \\
& = \frac{1}{2} \delta X_n^T \left(Q_{XX,n} + Q_{uX,n}^T K_n \right) \delta X_n
\end{aligned}
$$


これより、$\tilde{Q}_n(\bar{X} + \delta X_n , \bar{u}_n + \delta u_n^\star)$は以下のように構成できる。

$$
\begin{aligned}
\tilde{Q}_n(\bar{X} + \delta X_n , \bar{u}_n + \delta u_n^\star) =& \left(\bar{Q}_n + \frac{1}{2} Q_{u,n}^T \ k_n \right) +  \left(Q_{X,n} + Q_{uX,n}^T k_n \right)^T \delta X_n \\ 
&+ \frac{1}{2} \delta X_n^T \left(Q_{XX,n} + Q_{uX,n}^T K_n \right) \delta X_n
\end{aligned}
$$

よって、($k_n, K_n$)を用いて、価値関数 $\tilde{V}_n$の項は以下のようになる。このうち必要なのは、$k_{n-1},K_{n-1}$の構成に必要な$Q_{uu,n-1},Q_{uX,n-1},Q_{u,n-1}$であり、これらの構成に用いられている、$V_{X,n}, V_{XX,n}$である。

$$
\bar{V}_n = \bar{Q}_n + \frac{1}{2} Q_{u,n}^T \ k_n \\
$$

$$
\boxed{
V_{X,n} = Q_{X,n} + Q_{uX,n}^T k_n \\
}
$$

$$
\boxed{
V_{XX,n} = Q_{XX,n} + Q_{uX,n}^T K_n
}
$$

以上より、以下のようにbackward pass を構成することが出来る。$V_{X,n}$は$Q_{X,n}$を用いて、$V_{XX,n}$は$Q_{XX,n}$を用いて更新されているため、$Q$の項目に追加している。

$$
\begin{aligned}
(V_{XX,n+1}, V_{X, n+1}) \rightarrow& (Q_{X,n},Q_{uu,n},Q_{XX,n},Q_{u,n},Q_{uX,n}) \rightarrow (k_n, K_n)\\
\rightarrow (V_{XX,n}, V_{X, n}) \rightarrow& (Q_{X,n-1},Q_{uu,n-1},Q_{XX,n-1},Q_{u,n-1},Q_{uX,n-1}) \rightarrow (k_{n-1}, K_{n-1})  \rightarrow \\
 \vdots & \\ 
\rightarrow (V_{XX,1}, V_{X, 1}) \rightarrow& (Q_{X,0},Q_{uu,0},Q_{XX,0},Q_{u,0},Q_{uX,0}) \rightarrow (k_{0}, K_{0})
\end{aligned}
$$


## 3. backward pass の計算

Q関数は終端ではステージコストがないため、価値関数が終端コストと一致する。

$$
V_N(X_N) = \phi(X_N)
$$

終端での左辺の価値関数の二次近似式は以下となる。

$$
V_{N}(\bar{X}_{N} + \delta X_{N}) \approx \bar{V}_{N} + V_{X, N}^T \delta X_{N} + \frac{1}{2} \delta X_{N}^T \ V_{XX, N} \ \delta X_{N}
$$

右辺の終端コストの二次近似式は以下となる。

$$
\phi(\bar{X}_N + \delta X_N) \approx \phi(\bar{X}_N) + \phi_X^T \ \delta X_N + \frac{1}{2} \delta X_N^T \ \phi_{XX} \ \delta X_N
$$

よって価値関数を対応させると以下となる。

$$
V_{X,N} = \phi_X, \quad V_{XX, N} = \phi_{XX}
$$

よって、終端から次のように更新をしていけば、入力修正項の$k_n, K_n$の列を求めることが出来る。

$$
\begin{aligned}
(V_{XX,N}, V_{X, N}) \rightarrow& (Q_{X,N-1},Q_{u,N-1},Q_{uu,N-1},Q_{uX,N-1},Q_{XX,N-1}) \rightarrow (k_{N-1}, K_{N-1}) \rightarrow \\
 & \vdots \\
(V_{XX,n+1}, V_{X, n+1}) \rightarrow& (Q_{X,n},Q_{u,n},Q_{uu,n},Q_{uX,n},Q_{XX,n}) \rightarrow (k_{n}, K_{n})  \rightarrow \\
 \vdots & \\ (V_{XX,1}, V_{X, 1}) \rightarrow& (Q_{X,0},Q_{u,0},Q_{uu,0},Q_{uX,0},Q_{XX,0}) \rightarrow (k_{0}, K_{0})
\end{aligned}
$$


### backward pass の計算式

#### 終端 $N$

- 価値関数の係数

$$
\begin{aligned}
V_{X,N} &= \phi_X \\
V_{XX, N} &= \phi_{XX}
\end{aligned}
$$

- Q関数の係数

$$
\begin{aligned}
Q_{X,N-1} &= \ell_{X,N-1} + A_{N-1}^T V_{X,N} \\
Q_{u,N-1} &= \ell_{u,N-1} + B_{N-1}^T V_{X,N} \\
Q_{uu,N-1} &= \ell_{uu,N-1} + B_{N-1}^T V_{XX,N} B_{N-1} \\
Q_{uX,N-1} &= \ell_{uX, N-1} + B_{N-1}^T V_{XX,N} A_{N-1} \\
Q_{XX, N-1} &= \ell_{XX,N-1} + A_{N-1}^T V_{XX,N} A_{N-1} \\
\end{aligned}
$$

- 入力修正項の係数

$$
\begin{aligned}
k_{N-1} &= -Q_{uu,N-1}^{-1} \ Q_{u,N-1} \\
 K_{N-1} &= - Q_{uu,N-1}^{-1} \ Q_{uX,N-1}
\end{aligned}
$$

#### 時刻 $n$

- 価値関数の係数

$$
\begin{aligned}
V_{X,n+1} &= Q_{X, n+1} + Q_{uX, n+1} k_{n+1} \\
V_{XX, n+1} &= Q_{XX, n+1} + Q_{uX, n+1} K_{n+1}
\end{aligned}
$$

- Q関数の係数

$$
\begin{aligned}
Q_{X,n} &= \ell_{X,n} + A_{n}^T V_{X,n+1} \\
Q_{u,n} &= \ell_{u,n} + B_{n}^T V_{X,n+1} \\
Q_{uu,n} &= \ell_{uu,n} + B_{n}^T V_{XX,n+1} B_{n} \\
Q_{uX,n} &= \ell_{uX,n} + B_{n}^T V_{XX,n+1} A_{n} \\
Q_{XX,n} &= \ell_{XX,n} + A_{n}^T V_{XX,n+1} A_{n} \\
\end{aligned}
$$

- 入力修正項の係数

$$
\begin{aligned}
k_{n} &= -Q_{uu,n}^{-1} \ Q_{u,n} \\
 K_{n} &= - Q_{uu,n}^{-1} \ Q_{uX,n}
\end{aligned}
$$


